In [11]:
import xarray as xr
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
from datetime import datetime
import os

# =========================
# 出力フォルダ
# =========================
os.makedirs("csv", exist_ok=True)
os.makedirs("png", exist_ok=True)

# =========================
# 設定：十勝岳 1962 / 18:00 UTC
# =========================
target_date = "1962-06-29"          # UTC
date_str = "19620629"
target_time = "18:00"
hour_str = "18"

# Tokachidake (代表値)
lat0, lon0 = 43.418, 142.686
height = 2077.0  # [m] まずは山頂（噴出口標高にしたければここを変更）

z_target = 12000.0  # 火口からの噴煙高度 [m]

# ERA5 pressure-levels 取得ファイル名
ncfile = f"era5_tokachi_{date_str}_{hour_str}UTC_pl.nc"

# =========================
# 1) ERA5をダウンロード（初回だけ）
# =========================
if not os.path.exists(ncfile):
    import cdsapi
    c = cdsapi.Client()

    c.retrieve(
        "reanalysis-era5-pressure-levels",
        {
            "product_type": "reanalysis",
            "variable": [
                "u_component_of_wind",
                "v_component_of_wind",
                "temperature",
                "geopotential",
            ],
            "pressure_level": [str(p) for p in
                               [1000,975,950,925,900,875,850,825,800,775,750,725,700,
                                650,600,550,500,450,400,350,300,250,200,150,100,70,50]],
            "year": "1962",
            "month": "06",
            "day": "29",
            "time": ["18:00"],
            # [北, 西, 南, 東]
            "area": [44.5, 141.5, 42.5, 143.5],
            "format": "netcdf",
        },
        ncfile
    )

# =========================
# ERA5 読み込み（NetCDF）
# =========================
ncfile = f"era5_tokachi_{date_str}_{hour_str}UTC_pl.nc"
ds = xr.open_dataset(ncfile)

# 最近傍格子点
pt = ds.sel(latitude=lat0, longitude=lon0, method="nearest")

# 余計な次元が残る場合に備えて潰す（今回のファイル形式差にも強い）
for d in ["valid_time", "time", "expver", "number"]:
    if d in pt.dims:
        pt = pt.isel({d: 0}, drop=True)

pt = pt.squeeze(drop=True)

# pressure-level 座標名の揺れに対応
plev_name = "pressure_level" if "pressure_level" in pt.coords else "level"

# 必ず 1次元配列にする（pandas対策として堅牢）
pressure = np.ravel(pt[plev_name].values).astype(float)   # [hPa]
u = np.ravel(pt["u"].values)
v = np.ravel(pt["v"].values)
T = np.ravel(pt["t"].values)                              # [K]

g0 = 9.80665
altitude = np.ravel(pt["z"].values) / g0                  # geopotential -> [m]

# 風速・風向
wind_speed = np.sqrt(u**2 + v**2)
wind_dir = (np.degrees(np.arctan2(-u, -v)) + 360) % 360

# DataFrame
df_gfs = pd.DataFrame({
    "Pressure_hPa": pressure,
    "Altitude_m": altitude,
    "WindSpeed_mps": wind_speed,
    "WindDirection_deg": wind_dir,
    "Temperature_K": T
}).sort_values("Altitude_m")

# 保存（確認用）
df_gfs.to_csv(f"csv/era5_tokachi_{date_str}_{hour_str}UTC.csv", index=False)

# 火口基準（z=0 at vent altitude）
df_gfs["z_rel_m"] = df_gfs["Altitude_m"] - height
df_gfs = df_gfs.sort_values("z_rel_m").drop_duplicates(subset="z_rel_m")

# ---------- ★ここから追加：z=0アンカー点を作る ----------
# z_rel が負側最大（0に最も近い負）と、正側最小（0に最も近い正）を取る
df_neg = df_gfs[df_gfs["z_rel_m"] <= 0].copy()
df_pos = df_gfs[df_gfs["z_rel_m"] >= 0].copy()

if (len(df_neg) >= 1) and (len(df_pos) >= 1):
    a = df_neg.iloc[df_neg["z_rel_m"].values.argmax()]   # 0に最も近い負
    b = df_pos.iloc[df_pos["z_rel_m"].values.argmin()]   # 0に最も近い正

    za = float(a["z_rel_m"]); zb = float(b["z_rel_m"])
    # まれに za==zb（ほぼ起きないが念のため）
    if abs(zb - za) > 1e-6:
        w = (0.0 - za) / (zb - za)  # 0がa→bのどこにあるか

        def lin(xa, xb): 
            return float(xa + (xb - xa) * w)

        anchor = {
            "Pressure_hPa":       lin(a["Pressure_hPa"],       b["Pressure_hPa"]),
            "Altitude_m":         height,  # z_rel=0
            "WindSpeed_mps":      lin(a["WindSpeed_mps"],      b["WindSpeed_mps"]),
            "WindDirection_deg":  lin(a["WindDirection_deg"],  b["WindDirection_deg"]),
            "Temperature_K":      lin(a["Temperature_K"],      b["Temperature_K"]),
            "z_rel_m":            0.0,
        }

        # 既にz_rel=0が存在しない場合のみ追加
        if not np.any(np.isclose(df_gfs["z_rel_m"].values, 0.0, atol=1e-6)):
            df_gfs = pd.concat([df_gfs, pd.DataFrame([anchor])], ignore_index=True)

# 並べ直し（重要）
df_gfs = df_gfs.sort_values("z_rel_m").reset_index(drop=True)
# ---------- ★ここまで追加 ----------

# =========================
# 3) v(z), tempa(z), p(z)  ※外挿は上下端固定
# =========================
zmin = float(df_gfs["z_rel_m"].iloc[0])
zmax = float(df_gfs["z_rel_m"].iloc[-1])

vmin, vmax = float(df_gfs["WindSpeed_mps"].iloc[0]), float(df_gfs["WindSpeed_mps"].iloc[-1])
Tmin, Tmax = float(df_gfs["Temperature_K"].iloc[0]), float(df_gfs["Temperature_K"].iloc[-1])
pmin, pmax = float(df_gfs["Pressure_hPa"].iloc[0]) * 100.0, float(df_gfs["Pressure_hPa"].iloc[-1]) * 100.0

v_interp = interp1d(df_gfs["z_rel_m"], df_gfs["WindSpeed_mps"],
                    bounds_error=False, fill_value=(vmin, vmax))
T_interp = interp1d(df_gfs["z_rel_m"], df_gfs["Temperature_K"],
                    bounds_error=False, fill_value=(Tmin, Tmax))
p_interp = interp1d(df_gfs["z_rel_m"], df_gfs["Pressure_hPa"] * 100.0,
                    bounds_error=False, fill_value=(pmin, pmax))

def v_of_z(z):   return float(v_interp(z))
def tempa(z):    return float(T_interp(z))
def p(z):        return float(p_interp(z))

# =========================
# 4) 噴煙モデル（あなたのまま）
# =========================
n0 = 0.03
T0 = 1273.0
theta0 = np.deg2rad(90.0)
ke, kw = 0.06, 0.2
pi, g = np.pi, 9.8
rga, rgv = 285.0, 462.0
cpm, cpa = 1000.0, 1000.0
rhol = 2.5e3

pa = p(0)
nv = n0
z_stop = float(df_gfs["z_rel_m"].max())  # 例: ~18785 m

def rhoa(z): return p(z) / (rga * tempa(z))
def na(z, y1, Q): return 1.0 - Q / (pi * y1) if z > 0 else 0.0
def rg(z, y1, Q):
    na_ = na(z, y1, Q)
    return (na_ * rga + nv * (1 - na_) * rgv) / (na_ + nv * (1 - na_))
def cp(z, y1, Q): return na(z, y1, Q) * cpa + (1 - na(z, y1, Q)) * cpm
def temp(z, y1, y2, y3, Q): return 1.0 / cp(z, y1, Q) * (y3 / y1 - 0.5 * (y2 / y1)**2 - g * z)

def rho(z, y1, y2, y3, Q):
    na_ = na(z, y1, Q)
    return (((na_ + nv * (1 - na_)) * rg(z, y1, Q) * temp(z, y1, y2, y3, Q) / p(z))
            + (1 - na_) * (1 - nv) / rhol) ** (-1)

def r(z, y1, y2, y3, Q): return np.sqrt(y1 / rho(z, y1, y2, y3, Q) / (y2 / y1))

def uke(z, y1, y2, y4, Q):
    u_val = y2 / y1
    return ke * abs(u_val - v_of_z(z) * np.cos(y4)) + kw * abs(v_of_z(z) * np.sin(y4))

def f_vec(s, z, y, Q):
    y1, y2, y3, y4 = y
    r_ = r(z, y1, y2, y3, Q)
    rho_ = rho(z, y1, y2, y3, Q)
    rhoa_ = rhoa(z)
    uke_ = uke(z, y1, y2, y4, Q)
    f1 = 2 * uke_ * r_ * rhoa_
    f2 = r_**2 * (rhoa_ - rho_) * g * np.sin(y4) + v_of_z(z) * np.cos(y4) * f1
    f3 = (cpa * tempa(z) + g * z) * f1
    f4 = (r_**2 * (rhoa_ - rho_) * g * np.cos(y4) - v_of_z(z) * np.sin(y4) * f1) / y2
    return np.array([f1, f2, f3, f4])

def run_plume(r0, u0, T0, n0, ds_step=10.0, nstep=10000):
    """
    1回の噴煙計算を回して、最大到達高度 z_max を返す
    """
    # グローバルの v_of_z, tempa, p, などを使用
    # ただし n0, T0 はこの関数引数を優先したいので、内部で nv を作る
    nv_local = n0
    pa_local = p(0)

    def na_local(z, y1, Q):
        return 1.0 - Q / (pi * y1) if z > 0 else 0.0

    def rg_local(z, y1, Q):
        na_ = na_local(z, y1, Q)
        return (na_ * rga + nv_local * (1 - na_) * rgv) / (na_ + nv_local * (1 - na_))

    def cp_local(z, y1, Q):
        na_ = na_local(z, y1, Q)
        return na_ * cpa + (1 - na_) * cpm

    def temp_local(z, y1, y2, y3, Q):
        return 1.0 / cp_local(z, y1, Q) * (y3 / y1 - 0.5 * (y2 / y1)**2 - g * z)

    def rho_local(z, y1, y2, y3, Q):
        na_ = na_local(z, y1, Q)
        return (((na_ + nv_local * (1 - na_)) * rg_local(z, y1, Q) * temp_local(z, y1, y2, y3, Q) / p(z))
                + (1 - na_) * (1 - nv_local) / rhol) ** (-1)

    def r_local(z, y1, y2, y3, Q):
        return np.sqrt(y1 / rho_local(z, y1, y2, y3, Q) / (y2 / y1))

    def rhoa_local(z):
        return p(z) / (rga * tempa(z))

    def uke_local(z, y1, y2, y4, Q):
        u_val = y2 / y1
        return ke * abs(u_val - v_of_z(z) * np.cos(y4)) + kw * abs(v_of_z(z) * np.sin(y4))

    def f_vec_local(s, z, y, Q):
        y1, y2, y3, y4 = y
        r_ = r_local(z, y1, y2, y3, Q)
        rho_ = rho_local(z, y1, y2, y3, Q)
        rhoa_ = rhoa_local(z)
        uke_ = uke_local(z, y1, y2, y4, Q)
        f1 = 2 * uke_ * r_ * rhoa_
        f2 = r_**2 * (rhoa_ - rho_) * g * np.sin(y4) + v_of_z(z) * np.cos(y4) * f1
        f3 = (cpa * tempa(z) + g * z) * f1
        f4 = (r_**2 * (rhoa_ - rho_) * g * np.cos(y4) - v_of_z(z) * np.sin(y4) * f1) / y2
        return np.array([f1, f2, f3, f4])

    # 初期条件
    rho0 = (nv_local * rgv * T0 / pa_local + (1 - nv_local) / rhol) ** (-1)
    Q = rho0 * u0 * pi * r0**2

    y1 = Q / pi
    y2 = y1 * u0
    y3 = y1 * (cpm * T0 + u0**2 / 2)
    y4 = theta0
    y = np.array([y1, y2, y3, y4])

    s, z, x = 0.0, 0.0, 0.0
    z_list = []

    for _ in range(nstep):
        k1 = ds_step * f_vec_local(s, z, y, Q)
        k2 = ds_step * f_vec_local(s + ds_step/2, z + np.sin(y4)*ds_step/2, y + k1/2, Q)
        k3 = ds_step * f_vec_local(s + ds_step/2, z + np.sin(y4)*ds_step/2, y + k2/2, Q)
        k4 = ds_step * f_vec_local(s + ds_step,   z + np.sin(y4)*ds_step,   y + k3,   Q)
        y = y + (k1 + 2*k2 + 2*k3 + k4) / 6

        z += ds_step * np.sin(y4)
        x += ds_step * np.cos(y4)
        y4 = y[3]
        s += ds_step

        z_list.append(z)

        if (np.degrees(y4) <= 0) or (z > min(30000, z_stop)):
            break

    z_max = float(np.max(z_list)) if len(z_list) > 0 else 0.0
    return z_max, float(Q)

def solve_u0_for_target(r0, T0, n0, z_target,
                        u0_lo=30.0, u0_hi=400.0,
                        tol_z=50.0, max_iter=30):
    """
    z_max(u0) が単調増加に近い前提で u0 を二分探索。
    返り値: (best_u0, best_Q, best_zmax, success)
    """
    z_lo, Q_lo = run_plume(r0, u0_lo, T0, n0)
    z_hi, Q_hi = run_plume(r0, u0_hi, T0, n0)
    print(f"[DEBUG] r0={r0} T0={T0} n0={n0}  u0_lo={u0_lo} z_lo={z_lo:.1f}  u0_hi={u0_hi} z_hi={z_hi:.1f}")

    # 目標が bracket できない場合
    if not (z_lo <= z_target <= z_hi):
        return np.nan, np.nan, max(z_lo, z_hi), False

    best = None
    lo, hi = u0_lo, u0_hi
    for _ in range(max_iter):
        mid = 0.5 * (lo + hi)
        z_mid, Q_mid = run_plume(r0, mid, T0, n0)
        err = z_mid - z_target

        if (best is None) or (abs(err) < abs(best[2] - z_target)):
            best = (mid, Q_mid, z_mid)

        if abs(err) <= tol_z:
            return mid, Q_mid, z_mid, True

        if err < 0:
            lo = mid
        else:
            hi = mid

    # 収束しなくても最良を返す
    return best[0], best[1], best[2], True


# --- 探索範囲（まずは粗く） ---
#r0_grid = [50.0, 100.0, 150.0]  # m
#T0_grid = [1073.0, 1173.0, 1273.0, 1373.0]          # K（800,900,1000,1100°C）
#n0_grid = [0.01, 0.03, 0.05]                         # H2O（混合比の代表）
r0_grid = [10.0, 30.0, 50.0, 100.0, 150.0, 200.0]  # m
T0_grid = [1073.0, 1173.0, 1273.0, 1373.0]          # K（800,900,1000,1100°C）
n0_grid = [0.01, 0.02, 0.03, 0.04, 0.05]                         # H2O（混合比の代表）

rows = []
for r0 in r0_grid:
    for T0_try in T0_grid:
        for n0_try in n0_grid:
            u0_sol, Q_sol, z_sol, ok = solve_u0_for_target(
                r0=r0, T0=T0_try, n0=n0_try, z_target=z_target,
                u0_lo=30.0, u0_hi=400.0, tol_z=50.0, max_iter=30
            )
            rows.append({
                "r0_m": r0,
                "T0_K": T0_try,
                "n0": n0_try,
                "u0_mps": u0_sol,
                "Q_kgps": Q_sol,
                "zmax_m": z_sol,
                "ok": ok
            })

df_search = pd.DataFrame(rows)
df_search = df_search.sort_values(["ok", "Q_kgps"], ascending=[False, True]).reset_index(drop=True)

outcsv = f"csv/tokachi1962_fit_z{int(z_target)}m_{date_str}_{hour_str}UTC.csv"
df_search.to_csv(outcsv, index=False)
print("saved:", outcsv)

# 目標に近い上位を表示
print(df_search[df_search["ok"]].head(20))



/var/folders/xd/msh5srmj40702mpq6nhmmpjw0000gn/T/ipykernel_15219/688769081.py:242: RuntimeWarning: invalid value encountered in sqrt
  return np.sqrt(y1 / rho_local(z, y1, y2, y3, Q) / (y2 / y1))


[DEBUG] r0=10.0 T0=1073.0 n0=0.01  u0_lo=30.0 z_lo=nan  u0_hi=400.0 z_hi=11028.5
[DEBUG] r0=10.0 T0=1073.0 n0=0.02  u0_lo=30.0 z_lo=3316.4  u0_hi=400.0 z_hi=9957.1
[DEBUG] r0=10.0 T0=1073.0 n0=0.03  u0_lo=30.0 z_lo=2879.0  u0_hi=400.0 z_hi=9017.8
[DEBUG] r0=10.0 T0=1073.0 n0=0.04  u0_lo=30.0 z_lo=2606.4  u0_hi=400.0 z_hi=7421.5
[DEBUG] r0=10.0 T0=1073.0 n0=0.05  u0_lo=30.0 z_lo=2414.4  u0_hi=400.0 z_hi=6419.4
[DEBUG] r0=10.0 T0=1173.0 n0=0.01  u0_lo=30.0 z_lo=nan  u0_hi=400.0 z_hi=11132.0
[DEBUG] r0=10.0 T0=1173.0 n0=0.02  u0_lo=30.0 z_lo=3431.4  u0_hi=400.0 z_hi=10099.7
[DEBUG] r0=10.0 T0=1173.0 n0=0.03  u0_lo=30.0 z_lo=2974.3  u0_hi=400.0 z_hi=9219.6
[DEBUG] r0=10.0 T0=1173.0 n0=0.04  u0_lo=30.0 z_lo=2691.7  u0_hi=400.0 z_hi=7892.4
[DEBUG] r0=10.0 T0=1173.0 n0=0.05  u0_lo=30.0 z_lo=2491.5  u0_hi=400.0 z_hi=6713.4
[DEBUG] r0=10.0 T0=1273.0 n0=0.01  u0_lo=30.0 z_lo=4473.8  u0_hi=400.0 z_hi=11214.4
[DEBUG] r0=10.0 T0=1273.0 n0=0.02  u0_lo=30.0 z_lo=3519.6  u0_hi=400.0 z_hi=10199.8
[DEBU

In [18]:
import xarray as xr
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
from datetime import datetime
import os

# =========================
# 出力フォルダ
# =========================
os.makedirs("csv", exist_ok=True)
os.makedirs("png", exist_ok=True)

# =========================
# 設定：十勝岳 1962 / 18:00 UTC
# =========================
target_date = "1962-06-29"          # UTC
date_str = "19620629"
target_time = "18:00"
hour_str = "18"

# Tokachidake (代表値)
lat0, lon0 = 43.418, 142.686
height = 2077.0  # [m] まずは山頂（噴出口標高にしたければここを変更）

z_target = 12000.0  # 火口からの噴煙高度 [m]

# ERA5 pressure-levels 取得ファイル名
ncfile = f"era5_tokachi_{date_str}_{hour_str}UTC_pl.nc"

# =========================
# 1) ERA5をダウンロード（初回だけ）
# =========================
if not os.path.exists(ncfile):
    import cdsapi
    c = cdsapi.Client()

    c.retrieve(
        "reanalysis-era5-pressure-levels",
        {
            "product_type": "reanalysis",
            "variable": [
                "u_component_of_wind",
                "v_component_of_wind",
                "temperature",
                "geopotential",
            ],
            "pressure_level": [str(p) for p in
                               [1000,975,950,925,900,875,850,825,800,775,750,725,700,
                                650,600,550,500,450,400,350,300,250,200,150,100,70,50]],
            "year": "1962",
            "month": "06",
            "day": "29",
            "time": ["18:00"],
            # [北, 西, 南, 東]
            "area": [44.5, 141.5, 42.5, 143.5],
            "format": "netcdf",
        },
        ncfile
    )

# =========================
# ERA5 読み込み（NetCDF）
# =========================
ncfile = f"era5_tokachi_{date_str}_{hour_str}UTC_pl.nc"
ds = xr.open_dataset(ncfile)

# 最近傍格子点
pt = ds.sel(latitude=lat0, longitude=lon0, method="nearest")

# 余計な次元が残る場合に備えて潰す（今回のファイル形式差にも強い）
for d in ["valid_time", "time", "expver", "number"]:
    if d in pt.dims:
        pt = pt.isel({d: 0}, drop=True)

pt = pt.squeeze(drop=True)

# pressure-level 座標名の揺れに対応
plev_name = "pressure_level" if "pressure_level" in pt.coords else "level"

# 必ず 1次元配列にする（pandas対策として堅牢）
pressure = np.ravel(pt[plev_name].values).astype(float)   # [hPa]
u = np.ravel(pt["u"].values)
v = np.ravel(pt["v"].values)
T = np.ravel(pt["t"].values)                              # [K]

g0 = 9.80665
altitude = np.ravel(pt["z"].values) / g0                  # geopotential -> [m]

# 風速・風向
wind_speed = np.sqrt(u**2 + v**2)
wind_dir = (np.degrees(np.arctan2(-u, -v)) + 360) % 360

# DataFrame
df_gfs = pd.DataFrame({
    "Pressure_hPa": pressure,
    "Altitude_m": altitude,
    "WindSpeed_mps": wind_speed,
    "WindDirection_deg": wind_dir,
    "Temperature_K": T
}).sort_values("Altitude_m")

# 保存（確認用）
df_gfs.to_csv(f"csv/era5_tokachi_{date_str}_{hour_str}UTC.csv", index=False)

# 火口基準（z=0 at vent altitude）
df_gfs["z_rel_m"] = df_gfs["Altitude_m"] - height
df_gfs = df_gfs.sort_values("z_rel_m").drop_duplicates(subset="z_rel_m")

# ---------- ★ここから追加：z=0アンカー点を作る ----------
# z_rel が負側最大（0に最も近い負）と、正側最小（0に最も近い正）を取る
df_neg = df_gfs[df_gfs["z_rel_m"] <= 0].copy()
df_pos = df_gfs[df_gfs["z_rel_m"] >= 0].copy()

if (len(df_neg) >= 1) and (len(df_pos) >= 1):
    a = df_neg.iloc[df_neg["z_rel_m"].values.argmax()]   # 0に最も近い負
    b = df_pos.iloc[df_pos["z_rel_m"].values.argmin()]   # 0に最も近い正

    za = float(a["z_rel_m"]); zb = float(b["z_rel_m"])
    # まれに za==zb（ほぼ起きないが念のため）
    if abs(zb - za) > 1e-6:
        w = (0.0 - za) / (zb - za)  # 0がa→bのどこにあるか

        def lin(xa, xb): 
            return float(xa + (xb - xa) * w)

        anchor = {
            "Pressure_hPa":       lin(a["Pressure_hPa"],       b["Pressure_hPa"]),
            "Altitude_m":         height,  # z_rel=0
            "WindSpeed_mps":      lin(a["WindSpeed_mps"],      b["WindSpeed_mps"]),
            "WindDirection_deg":  lin(a["WindDirection_deg"],  b["WindDirection_deg"]),
            "Temperature_K":      lin(a["Temperature_K"],      b["Temperature_K"]),
            "z_rel_m":            0.0,
        }

        # 既にz_rel=0が存在しない場合のみ追加
        if not np.any(np.isclose(df_gfs["z_rel_m"].values, 0.0, atol=1e-6)):
            df_gfs = pd.concat([df_gfs, pd.DataFrame([anchor])], ignore_index=True)

# 並べ直し（重要）
df_gfs = df_gfs.sort_values("z_rel_m").reset_index(drop=True)
# ---------- ★ここまで追加 ----------

# =========================
# 3) v(z), tempa(z), p(z)  ※外挿は上下端固定
# =========================
zmin = float(df_gfs["z_rel_m"].iloc[0])
zmax = float(df_gfs["z_rel_m"].iloc[-1])

vmin, vmax = float(df_gfs["WindSpeed_mps"].iloc[0]), float(df_gfs["WindSpeed_mps"].iloc[-1])
Tmin, Tmax = float(df_gfs["Temperature_K"].iloc[0]), float(df_gfs["Temperature_K"].iloc[-1])
pmin, pmax = float(df_gfs["Pressure_hPa"].iloc[0]) * 100.0, float(df_gfs["Pressure_hPa"].iloc[-1]) * 100.0

v_interp = interp1d(df_gfs["z_rel_m"], df_gfs["WindSpeed_mps"],
                    bounds_error=False, fill_value=(vmin, vmax))
T_interp = interp1d(df_gfs["z_rel_m"], df_gfs["Temperature_K"],
                    bounds_error=False, fill_value=(Tmin, Tmax))
p_interp = interp1d(df_gfs["z_rel_m"], df_gfs["Pressure_hPa"] * 100.0,
                    bounds_error=False, fill_value=(pmin, pmax))

def v_of_z(z):   return float(v_interp(z))
def tempa(z):    return float(T_interp(z))
def p(z):        return float(p_interp(z))

# =========================
# 4) 噴煙モデル（あなたのまま）
# =========================
n0 = 0.03
T0 = 1273.0
theta0 = np.deg2rad(90.0)
ke, kw = 0.06, 0.2
pi, g = np.pi, 9.8
rga, rgv = 285.0, 462.0
cpm, cpa = 1000.0, 1000.0
rhol = 2.5e3

pa = p(0)
nv = n0
z_stop = float(df_gfs["z_rel_m"].max())  # 例: ~18785 m

def rhoa(z): return p(z) / (rga * tempa(z))
def na(z, y1, Q): return 1.0 - Q / (pi * y1) if z > 0 else 0.0
def rg(z, y1, Q):
    na_ = na(z, y1, Q)
    return (na_ * rga + nv * (1 - na_) * rgv) / (na_ + nv * (1 - na_))
def cp(z, y1, Q): return na(z, y1, Q) * cpa + (1 - na(z, y1, Q)) * cpm
def temp(z, y1, y2, y3, Q): return 1.0 / cp(z, y1, Q) * (y3 / y1 - 0.5 * (y2 / y1)**2 - g * z)

def rho(z, y1, y2, y3, Q):
    na_ = na(z, y1, Q)
    return (((na_ + nv * (1 - na_)) * rg(z, y1, Q) * temp(z, y1, y2, y3, Q) / p(z))
            + (1 - na_) * (1 - nv) / rhol) ** (-1)

def r(z, y1, y2, y3, Q): return np.sqrt(y1 / rho(z, y1, y2, y3, Q) / (y2 / y1))

def uke(z, y1, y2, y4, Q):
    u_val = y2 / y1
    return ke * abs(u_val - v_of_z(z) * np.cos(y4)) + kw * abs(v_of_z(z) * np.sin(y4))

def f_vec(s, z, y, Q):
    y1, y2, y3, y4 = y
    r_ = r(z, y1, y2, y3, Q)
    rho_ = rho(z, y1, y2, y3, Q)
    rhoa_ = rhoa(z)
    uke_ = uke(z, y1, y2, y4, Q)
    f1 = 2 * uke_ * r_ * rhoa_
    f2 = r_**2 * (rhoa_ - rho_) * g * np.sin(y4) + v_of_z(z) * np.cos(y4) * f1
    f3 = (cpa * tempa(z) + g * z) * f1
    f4 = (r_**2 * (rhoa_ - rho_) * g * np.cos(y4) - v_of_z(z) * np.sin(y4) * f1) / y2
    return np.array([f1, f2, f3, f4])

def run_plume(r0, u0, T0, n0, ds_step=10.0, nstep=10000):
    """
    1回の噴煙計算を回して、最大到達高度 z_max を返す
    """
    # グローバルの v_of_z, tempa, p, などを使用
    # ただし n0, T0 はこの関数引数を優先したいので、内部で nv を作る
    nv_local = n0
    pa_local = p(0)

    def na_local(z, y1, Q):
        return 1.0 - Q / (pi * y1) if z > 0 else 0.0

    def rg_local(z, y1, Q):
        na_ = na_local(z, y1, Q)
        return (na_ * rga + nv_local * (1 - na_) * rgv) / (na_ + nv_local * (1 - na_))

    def cp_local(z, y1, Q):
        na_ = na_local(z, y1, Q)
        return na_ * cpa + (1 - na_) * cpm

    def temp_local(z, y1, y2, y3, Q):
        return 1.0 / cp_local(z, y1, Q) * (y3 / y1 - 0.5 * (y2 / y1)**2 - g * z)

    def rho_local(z, y1, y2, y3, Q):
        na_ = na_local(z, y1, Q)
        return (((na_ + nv_local * (1 - na_)) * rg_local(z, y1, Q) * temp_local(z, y1, y2, y3, Q) / p(z))
                + (1 - na_) * (1 - nv_local) / rhol) ** (-1)

    def r_local(z, y1, y2, y3, Q):
        return np.sqrt(y1 / rho_local(z, y1, y2, y3, Q) / (y2 / y1))

    def rhoa_local(z):
        return p(z) / (rga * tempa(z))

    def uke_local(z, y1, y2, y4, Q):
        u_val = y2 / y1
        return ke * abs(u_val - v_of_z(z) * np.cos(y4)) + kw * abs(v_of_z(z) * np.sin(y4))

    def f_vec_local(s, z, y, Q):
        y1, y2, y3, y4 = y
        r_ = r_local(z, y1, y2, y3, Q)
        rho_ = rho_local(z, y1, y2, y3, Q)
        rhoa_ = rhoa_local(z)
        uke_ = uke_local(z, y1, y2, y4, Q)
        f1 = 2 * uke_ * r_ * rhoa_
        f2 = r_**2 * (rhoa_ - rho_) * g * np.sin(y4) + v_of_z(z) * np.cos(y4) * f1
        f3 = (cpa * tempa(z) + g * z) * f1
        f4 = (r_**2 * (rhoa_ - rho_) * g * np.cos(y4) - v_of_z(z) * np.sin(y4) * f1) / y2
        return np.array([f1, f2, f3, f4])

    # 初期条件
    rho0 = (nv_local * rgv * T0 / pa_local + (1 - nv_local) / rhol) ** (-1)
    Q = rho0 * u0 * pi * r0**2

    y1 = Q / pi
    y2 = y1 * u0
    y3 = y1 * (cpm * T0 + u0**2 / 2)
    y4 = theta0
    y = np.array([y1, y2, y3, y4])

    s, z, x = 0.0, 0.0, 0.0
    z_list = []

    for _ in range(nstep):
        k1 = ds_step * f_vec_local(s, z, y, Q)
        k2 = ds_step * f_vec_local(s + ds_step/2, z + np.sin(y4)*ds_step/2, y + k1/2, Q)
        k3 = ds_step * f_vec_local(s + ds_step/2, z + np.sin(y4)*ds_step/2, y + k2/2, Q)
        k4 = ds_step * f_vec_local(s + ds_step,   z + np.sin(y4)*ds_step,   y + k3,   Q)
        y = y + (k1 + 2*k2 + 2*k3 + k4) / 6

        z += ds_step * np.sin(y4)
        x += ds_step * np.cos(y4)
        y4 = y[3]
        s += ds_step

        z_list.append(z)

        if (np.degrees(y4) <= 0) or (z > min(30000, z_stop)):
            break

    z_max = float(np.max(z_list)) if len(z_list) > 0 else 0.0
    return z_max, float(Q)

def solve_u0_for_target_downscan(
    r0, T0, n0, z_target,
    u0_hi=400.0,
    u0_floor=5.0,          # これ以下は探索しない
    down_factor=0.85,      # 1回で 20% 下げる（0.9でもOK）
    tol_z=50.0,
    max_bisect=35,
    debug=False,
):
    """
    上限 u0_hi から下げていき、(噴煙成立して z>=target) と (噴煙不成立 or z<target) の境界を探す。
    返り値: (best_u0, best_Q, best_zmax, success)
    """

    def plume_ok(z):
        # 噴煙として成立した＆数値が壊れてない
        return np.isfinite(z) and (z > 0.0)

    # 1) まず上限でチェック
    z_hi, Q_hi = run_plume(r0, u0_hi, T0, n0)
    if debug:
        print(f"[DEBUG-HI] r0={r0} T0={T0} n0={n0} u0_hi={u0_hi} z_hi={z_hi}")

    if (not plume_ok(z_hi)) or (z_hi < z_target):
        # 上限でも届かない/成立しないなら、これ以上どうにもならない
        return np.nan, np.nan, (z_hi if np.isfinite(z_hi) else np.nan), False

    # 2) u0 を下げて、初めて「届かない or 不成立」になる点を探す
    u_good = u0_hi
    z_good = z_hi
    Q_good = Q_hi

    u_bad = None
    z_bad = None

    u = u0_hi
    while True:
        u_next = u * down_factor
        if u_next < u0_floor:
            # 下限まで下げても target を下回らない（=この範囲では常に届く）
            # → 最小u0はこの探索では決められないので、u0_floorを返すか、False扱いにするか選べる
            z_floor, Q_floor = run_plume(r0, u0_floor, T0, n0)
            if plume_ok(z_floor) and (z_floor >= z_target):
                # u0_floor でも届く：この範囲内では「最小u0」を特定できない
                return u0_floor, Q_floor, z_floor, True
            else:
                # u0_floor で落ちるなら bracket はできてる
                u_bad, z_bad = u0_floor, z_floor
                break

        z_next, Q_next = run_plume(r0, u_next, T0, n0)

        if debug:
            print(f"[DEBUG-DOWN] u={u_next:.3f} z={z_next}")

        if plume_ok(z_next) and (z_next >= z_target):
            # まだ届く → good を更新してさらに下げる
            u_good, z_good, Q_good = u_next, z_next, Q_next
            u = u_next
            continue

        # 届かない or 不成立(NaN含む) → ここで境界を挟めた
        u_bad, z_bad = u_next, z_next
        break

    # 3) bracket: u_good(届く) と u_bad(届かない/不成立) の間で二分探索
    lo = min(u_good, u_bad)
    hi = max(u_good, u_bad)

    # 注意：hi 側が "good" であることを保証する
    # ここでは u_good が good なので、hi= max(...) が good とは限らない → 判定して入替
    # good: z>=target and finite
    z_lo, _ = run_plume(r0, lo, T0, n0)
    z_hi2, _ = run_plume(r0, hi, T0, n0)

    if plume_ok(z_hi2) and (z_hi2 >= z_target):
        u_hi_good = hi
        u_lo_bad = lo
    else:
        u_hi_good = lo
        u_lo_bad = hi

    best_u, best_Q, best_z = u_hi_good, Q_good, z_good

    for _ in range(max_bisect):
        mid = 0.5 * (u_lo_bad + u_hi_good)
        z_mid, Q_mid = run_plume(r0, mid, T0, n0)

        # mid が good なら、さらに下（小さいu0）へ
        if plume_ok(z_mid) and (z_mid >= z_target):
            u_hi_good = mid
            best_u, best_Q, best_z = mid, Q_mid, z_mid
        else:
            # bad 側へ
            u_lo_bad = mid

        # 収束判定：z で判定（bad側はNaNあり得るので、good側の誤差で見る）
        if plume_ok(best_z) and abs(best_z - z_target) <= tol_z:
            return best_u, best_Q, best_z, True

        # u の幅でも止める
        if abs(u_hi_good - u_lo_bad) < 1e-3:
            break

#    return best_u, best_Q, best_z, True
    converged = (np.isfinite(best_z) and abs(best_z - z_target) <= tol_z)
    return best_u, best_Q, best_z, converged

# --- 探索範囲（まずは粗く） ---
r0_grid = [30.0, 40.0, 50.0, 60.0, 70.0, 80.0]  # m
T0_grid = [1073.0, 1173.0, 1273.0, 1373.0]          # K（800,900,1000,1100°C）
n0_grid = [0.03, 0.04, 0.05]                         # H2O（混合比の代表）
#r0_grid = [10.0, 30.0, 50.0, 100.0, 150.0, 200.0]  # m
#T0_grid = [1073.0, 1173.0, 1273.0, 1373.0]          # K（800,900,1000,1100°C）
#n0_grid = [0.01, 0.02, 0.03, 0.04, 0.05]                         # H2O（混合比の代表）

rows = []
for r0 in r0_grid:
    for T0_try in T0_grid:
        for n0_try in n0_grid:
            u0_sol, Q_sol, z_sol, ok = solve_u0_for_target_downscan(
            r0=r0, T0=T0_try, n0=n0_try, z_target=z_target,
            u0_hi=400.0,
            u0_floor=5.0,
            down_factor=0.85,   # 0.8〜0.9の間で調整
            tol_z=50.0,
            max_bisect=35,
            debug=False
            )
            rows.append({
                "r0_m": r0,
                "T0_K": T0_try,
                "n0": n0_try,
                "u0_mps": u0_sol,
                "Q_kgps": Q_sol,
                "zmax_m": z_sol,
                "ok": ok
            })

df_search = pd.DataFrame(rows)
df_search = df_search.sort_values(["ok", "Q_kgps"], ascending=[False, True]).reset_index(drop=True)

outcsv = f"csv/tokachi1962_fit_z{int(z_target)}m_{date_str}_{hour_str}UTC.csv"
df_search.to_csv(outcsv, index=False)
print("saved:", outcsv)

# 目標に近い上位を表示
#print(df_search[df_search["ok"]].head(20))
# 目標に近い全てを表示
df_search["dz_m"] = abs(df_search["zmax_m"] - z_target)
print(df_search[df_search["ok"]].sort_values("dz_m"))


/var/folders/xd/msh5srmj40702mpq6nhmmpjw0000gn/T/ipykernel_15219/1791680325.py:242: RuntimeWarning: invalid value encountered in sqrt
  return np.sqrt(y1 / rho_local(z, y1, y2, y3, Q) / (y2 / y1))


saved: csv/tokachi1962_fit_z12000m_19620629_18UTC.csv
    r0_m    T0_K    n0      u0_mps        Q_kgps        zmax_m    ok  \
46  60.0  1073.0  0.04   82.224016  3.691210e+06  12000.011404  True   
47  50.0  1073.0  0.03   89.172524  3.704653e+06  12004.907022  True   
34  60.0  1173.0  0.05   96.734136  3.179246e+06  12006.097618  True   
11  40.0  1373.0  0.05  200.972406  2.508393e+06  12006.673578  True   
31  60.0  1173.0  0.03   56.896703  3.114181e+06  12007.580571  True   
16  70.0  1273.0  0.04   52.629450  2.711239e+06  12008.012239  True   
26  30.0  1273.0  0.04  301.750000  2.855175e+06  12008.064064  True   
48  70.0  1073.0  0.05   75.796646  3.706308e+06  12008.872565  True   
17  80.0  1273.0  0.05   50.495824  2.718846e+06  12009.051942  True   
51  60.0  1073.0  0.05  104.908852  3.768861e+06  12010.207987  True   
2   80.0  1373.0  0.05   48.362197  2.414489e+06  12011.645614  True   
8   40.0  1373.0  0.03  118.613523  2.465782e+06  12012.167809  True   
5   50.0  